# 00 · Preparación del corpus de fragmentos

**Etapa de CRISP-DM:** preparación de los datos. Alimenta el **objetivo específico 1**
—caracterizar el corpus y definir los requerimientos de su preparación— y construye la
unidad de análisis que usan los objetivos 2 y 3.

Parte de las 2.432 reseñas en español ya seleccionadas en el proyecto previo
`reputacion-popayan` (solo lectura) y produce `data/processed/fragmentos.csv`.

**La unidad de análisis es el fragmento**: la reseña partida en cláusulas, es decir la
oración más los cortes en conectores adversativos. Cada fragmento conserva el
`review_id` de su reseña de origen.

Dos decisiones separan este trabajo del original y se validan más abajo:

1. Se **re-segmentan** las reseñas en vez de reutilizar `sentimiento_fragmentos.csv`,
   que solo guarda las cláusulas que activan alguna dimensión y está sesgado hacia el
   vocabulario del diccionario.
2. Las dimensiones se **recalculan sobre cada fragmento**, no se heredan de la reseña.

> ### Versión Colab de `00_preparacion`
>
> Equivalente en contenido a la versión local `notebooks/00_preparacion.ipynb`. Solo cambian
> las dos primeras celdas, que instalan dependencias y resuelven de dónde salen los
> datos. **Ejecuta las dos y sigue de largo**; el resto del notebook es idéntico.
>
> El notebook **no recalcula nada costoso**: lee de `data/processed/` los resultados ya
> producidos por los scripts de `scripts/`. Necesita, por tanto, que el proyecto viaje
> con su carpeta `data/processed/`.

In [1]:
# 1/2 · Dependencias. En local no hace falta: el entorno de uv ya las tiene.
import sys


def en_colab() -> bool:
    # `find_spec` no sirve aquí: lanza ModuleNotFoundError si no existe `google`
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


EN_COLAB = en_colab()

if EN_COLAB:
    %pip install -q pandas numpy pyarrow scikit-learn matplotlib sentence-transformers bertopic gensim
    print("dependencias instaladas")
else:
    print("no se detectó Colab: se asume el entorno de uv, no se instala nada")

no se detectó Colab: se asume el entorno de uv, no se instala nada


In [2]:
# 2/2 · Acceso a los datos. Tres caminos, en orden de preferencia:
#   a) el proyecto ya está en Drive          -> se monta y se usa
#   b) se sube un .zip del proyecto          -> se descomprime
#   c) se está ejecutando en local           -> se usa la carpeta del repositorio
import sys
import zipfile
from pathlib import Path

CARPETA_EN_DRIVE = "MyDrive/topicos-popayan"   # ajusta si la guardaste en otra ruta


def _valida(raiz):
    return (Path(raiz) / "src" / "config.py").exists() and (
        Path(raiz) / "data" / "processed" / "fragmentos.csv"
    ).exists()


def resolver_raiz():
    if not EN_COLAB:
        aqui = Path.cwd()
        for candidata in (aqui, aqui.parent, aqui.parent.parent):
            if _valida(candidata):
                return candidata
        raise RuntimeError("No se encontró la raíz del proyecto desde el directorio actual")

    from google.colab import drive, files

    try:
        drive.mount("/content/drive", force_remount=False)
        candidata = Path("/content/drive") / CARPETA_EN_DRIVE
        if _valida(candidata):
            return candidata
        print(f"No se encontró el proyecto en {candidata}")
    except Exception as error:                      # Drive no disponible o rechazado
        print(f"No se pudo montar Drive ({error})")

    print("Sube un .zip con el proyecto completo (debe incluir src/ y data/processed/)")
    subidos = files.upload()
    nombre = next(iter(subidos))
    destino = Path("/content/proyecto")
    with zipfile.ZipFile(nombre) as archivo:
        archivo.extractall(destino)
    candidatas = [destino, *(p.parent for p in destino.rglob("src/config.py"))]
    for candidata in candidatas:
        if _valida(candidata):
            return candidata
    raise RuntimeError("El zip no contiene src/ y data/processed/ juntos")


RAIZ = resolver_raiz()
sys.path.insert(0, str(RAIZ))
print(f"raíz del proyecto: {RAIZ.name}/")

raíz del proyecto: topicos-popayan/


In [3]:
# El módulo de rutas deduce la raíz de su propia ubicación, así que ya apunta bien
from src.config import DATOS_PROCESADOS, FIGURAS

faltan = [n for n in ("fragmentos.csv", "fragmentos_topicos.csv", "topicos_etiquetados.csv")
          if not (DATOS_PROCESADOS / n).exists()]
print("data/processed:", "completo" if not faltan else f"FALTAN {faltan}")
print("figuras:", "sí" if FIGURAS.exists() else "no encontradas")

data/processed: completo
figuras: sí


In [4]:
import pandas as pd

from src.config import DIMENSIONES, DATOS_PROCESADOS, MANIFIESTO
from src.diccionario import cargar_diccionario
from src.preparacion import nombres_establecimientos, patron_nombres_completos, patrones_para
from src.preparacion import enmascarar_locales, texto_para_ctfidf
from src.segmentacion import LARGO_MINIMO_FRAGMENTO, segmentar

pd.set_option("display.max_colwidth", 95)

fragmentos = pd.read_csv(DATOS_PROCESADOS / "fragmentos.csv")
print(f"{len(fragmentos)} fragmentos · {fragmentos['review_id'].nunique()} reseñas")

5913 fragmentos · 2430 reseñas


## 1. Trazabilidad de lo heredado

Cada archivo leído del proyecto original queda registrado con su número de filas y su
hash SHA-256. Si alguno cambiara, el hash lo delata y el análisis deja de ser
reproducible sin que nadie se entere.

In [5]:
pd.read_csv(MANIFIESTO)

,ruta,filas,columnas,hash_sha256,bytes
0,data/raw/corpus_final.csv,8451,15,8478fa5ace844b13c3dcc239f753e327af52e1bc607590e5dddba2c2f90c6fbd,1979504
1,data/raw/marco_muestral_final.csv,133,19,2c4a8587aed84d48dc42b2d8928e14e6c54a332af546daf9d385bdf0069ec175,32933
2,data/interim/idioma_resenas.csv,4102,7,0a567e58e490004136a61b01480713fc899b8b1f861035893d48c5e200856bd2,474200
3,data/interim/dimensiones_aptas_es.csv,2432,11,52a10c944a62c9a282aa6571905943dbf6f261930249f984086bf76fc598702b,309996
4,data/interim/sentimiento_fragmentos.csv,6562,4,bee608b5665b9af09f7ece55edc2371d1142562103e8a7c3c31718659ffb5e2e,1117891
5,diccionario/dimensiones.csv,144,8,67393d3a4e7cf1ff0187bdad49a5ecf9b6e6beec7dcd5fe12fb2ea35124a21df,15231


## 2. Embudo: de 8.451 reseñas a 5.913 fragmentos

El filtro de idioma y el de largo mínimo vienen del proyecto original y no se
recalculan. La única pérdida que introduce este trabajo son los 2 duplicados reales.

Ojo con la lectura: **la unidad cambia a mitad del embudo**. Las 2.430 reseñas producen
5.913 fragmentos porque cada reseña se parte en 2,4 cláusulas en promedio.

In [6]:
embudo = pd.DataFrame(
    [
        ("Reseñas del corpus heredado", 8451, "reseñas"),
        ("Con texto y ≥ 50 caracteres", 2645, "reseñas"),
        ("En español", 2432, "reseñas"),
        ("Sin duplicado real", 2430, "reseñas"),
        ("Fragmentos por cláusula", len(fragmentos), "fragmentos"),
    ],
    columns=["etapa", "n", "unidad"],
)
embudo

,etapa,n,unidad
0,Reseñas del corpus heredado,8451,reseñas
1,Con texto y ≥ 50 caracteres,2645,reseñas
2,En español,2432,reseñas
3,Sin duplicado real,2430,reseñas
4,Fragmentos por cláusula,5913,fragmentos


### Los duplicados, y una limitación que hay que declarar

Se elimina solo el duplicado real: misma persona y mismo texto. **El corpus heredado no
trae identificador de autor**, solo `autor_n_resenas`, que es el único proxy disponible.
Es débil en ambas direcciones y afecta a 2 filas de 2.432, así que no altera ningún
resultado, pero la deduplicación no debe presentarse como exacta.

## 3. Segmentación por cláusula

La regla se copió sin cambios de `src/sentimiento.py` del proyecto original, para que
los resultados sigan siendo comparables entre los dos trabajos. Corta en puntuación
`.!?;¡¿`, salto de línea, `<br>` y los conectores adversativos (*pero, aunque, sin
embargo, no obstante, eso sí, lo malo, lo único, mientras que*), que **se consumen en el
corte**. Descarta los fragmentos de menos de 10 caracteres.

El corte en conectores es lo que separa opiniones de signo opuesto dentro de una misma
oración, que es justo donde la reseña dice dos cosas distintas.

In [7]:
ejemplos = [
    "Un lugar pequeño pero con comida deliciosa",
    "La comida deliciosa. El servicio muy lento.",
    "Comida rica, aunque el precio es alto. Ok.",
]
for texto in ejemplos:
    print(f"{texto!r}\n   -> {segmentar(texto)}\n")
print(f"largo mínimo aplicado: {LARGO_MINIMO_FRAGMENTO} caracteres")

'Un lugar pequeño pero con comida deliciosa'
   -> ['Un lugar pequeño', 'con comida deliciosa']

'La comida deliciosa. El servicio muy lento.'
   -> ['La comida deliciosa', 'El servicio muy lento']

'Comida rica, aunque el precio es alto. Ok.'
   -> ['Comida rica', 'el precio es alto']

largo mínimo aplicado: 10 caracteres


## 4. Enmascarado de nombres de establecimiento

Los nombres propios se reemplazan por `[LOCAL]` para que el modelo agrupe por *lo que se
dice* y no por *de quién se habla*. La búsqueda es insensible a mayúsculas y tildes,
pero el reemplazo se hace sobre el texto original, que conserva sus tildes.

Tres reglas evitan que el enmascarado se coma texto legítimo. Sin ellas afectaba al
14,0 % de los fragmentos por falsos positivos; con ellas, al 1,8 %:

- **Los nombres de un solo token no se enmascaran.** «Carantanta» es a la vez un local y
  un plato patrimonial.
- **Para el local de la propia reseña** se aceptan abreviaturas; **para los demás** se
  exige el nombre completo.
- **La secuencia debe aportar algo propio del local**: se descartan las que son solo
  palabras funcionales (`de la`, de «El Fogón de la Abuela») y las que son funcionales
  más un término del diccionario (`de comidas`, de «Plazoleta de Comidas»), que borrarían
  justo la señal que se quiere medir.

In [8]:
patron = patron_nombres_completos(nombres_establecimientos())
casos = [
    ("En La Cosecha probé el salpicón payanés", "La Cosecha Parrillada"),
    ("la calidad de la comida es buena", "El Fogón de la Abuela"),
    ("amplia variedad de comidas", "Plazoleta de Comidas"),
]
for texto, propio in casos:
    salida, n = enmascarar_locales(texto, patrones_para(propio, patron))
    print(f"{texto!r}\n   local: {propio!r}\n   -> {salida!r}  ({n} marcas)\n")

'En La Cosecha probé el salpicón payanés'
   local: 'La Cosecha Parrillada'
   -> 'En [LOCAL] probé el salpicón payanés'  (1 marcas)

'la calidad de la comida es buena'
   local: 'El Fogón de la Abuela'
   -> 'la calidad de la comida es buena'  (0 marcas)

'amplia variedad de comidas'
   local: 'Plazoleta de Comidas'
   -> 'amplia variedad de comidas'  (0 marcas)



## 5. Dos textos con propósitos distintos

- **`texto`** es el fragmento enmascarado pero natural: conserva mayúsculas, tildes y
  stopwords. Va a los **embeddings**, porque el modelo de lenguaje espera texto tal como
  se escribe.
- **`texto_limpio`** es minúsculas, sin puntuación y sin stopwords. Solo alimenta el
  **c-TF-IDF**, que nombra los tópicos.

Las **negaciones se conservan** en `texto_limpio`. Quitarlas, como haría una lista de
stopwords estándar, etiquetaba un tópico con su contrario exacto: el tópico de «no
recomiendo», con rating medio 1,53, aparecía rotulado «recomiendo».

In [9]:
for texto in ["No lo recomiendo", "Lo recomiendo mucho", "¡Comida DELICIOSA, excelente!"]:
    print(f"{texto!r:<36} -> {texto_para_ctfidf(texto)!r}")

'No lo recomiendo'                   -> 'no recomiendo'
'Lo recomiendo mucho'                -> 'recomiendo'
'¡Comida DELICIOSA, excelente!'      -> 'comida deliciosa excelente'


## 6. Validación: las etiquetas por fragmento reproducen la medición del original

Esta es la comprobación que autoriza a recalcular las dimensiones por fragmento en vez
de heredarlas. Si se agregan de vuelta a nivel de reseña con un `any` por `review_id`,
deben coincidir con la medición del proyecto original, que se hizo sobre el texto
completo y está validada.

In [10]:
from src.config import DIMENSIONES_APTAS_ES

heredada = pd.read_csv(DIMENSIONES_APTAS_ES)
heredada = heredada[heredada["review_id"].isin(fragmentos["review_id"])]

comparacion = pd.DataFrame({
    "heredada": [int(heredada[d].sum()) for d in DIMENSIONES],
    "recalculada": [int(fragmentos.groupby("review_id")[d].any().sum()) for d in DIMENSIONES],
}, index=list(DIMENSIONES))
comparacion["diferencia"] = comparacion["recalculada"] - comparacion["heredada"]
comparacion["pct_diferencia"] = (100 * comparacion["diferencia"] / comparacion["heredada"]).round(2)
comparacion.sort_values("heredada", ascending=False)

,heredada,recalculada,diferencia,pct_diferencia
comida,1883,1878,-5,-0.27
servicio,1417,1416,-1,-0.07
ambiente,992,982,-10,-1.01
precio,431,431,0,0.00
espera,374,374,0,0.00
patrimonio,244,243,-1,-0.41


**La diferencia máxima es de 10 reseñas en `ambiente`, un 1,0 %**, y se explica por las
2 reseñas eliminadas y por el enmascarado. Recalcular sobre cláusulas no pierde señal,
así que la decisión queda validada.

A nivel de fragmento las cifras son mucho más bajas porque cambia el denominador: una
reseña de tres cláusulas habla de comida en una sola de ellas.

In [11]:
prevalencia = pd.DataFrame({
    "fragmentos": [int(fragmentos[d].sum()) for d in DIMENSIONES],
    "pct_fragmentos": [round(100 * fragmentos[d].mean(), 1) for d in DIMENSIONES],
    "pct_resenas": [round(100 * fragmentos.groupby("review_id")[d].any().mean(), 1)
                    for d in DIMENSIONES],
}, index=list(DIMENSIONES))
prevalencia.sort_values("fragmentos", ascending=False)

,fragmentos,pct_fragmentos,pct_resenas
comida,2555,43.2,77.3
servicio,1700,28.8,58.3
ambiente,1109,18.8,40.4
precio,468,7.9,17.7
espera,415,7.0,15.4
patrimonio,291,4.9,10.0


## 7. Dimensiones activas por fragmento

El AMI del objetivo 3 se calcula solo sobre fragmentos con **exactamente una** dimensión
activa. Esta tabla dice cuánto corpus queda disponible para esa medida.

In [12]:
reparto = fragmentos["n_dim"].value_counts().sort_index().to_frame("fragmentos")
reparto["%"] = (100 * reparto["fragmentos"] / len(fragmentos)).round(1)
reparto.index.name = "n_dim"
print(reparto.to_string())
print(f"\ndisponible para AMI: {(fragmentos['n_dim'] == 1).sum()} fragmentos "
      f"({100 * (fragmentos['n_dim'] == 1).mean():.1f} %)")
print(f"sin ninguna dimensión: {(fragmentos['n_dim'] == 0).sum()} "
      f"({100 * (fragmentos['n_dim'] == 0).mean():.1f} %)")

       fragmentos     %
n_dim                  
0            1722  29.1
1            2483  42.0
2            1156  19.6
3             470   7.9
4              77   1.3
5               5   0.1

disponible para AMI: 2483 fragmentos (42.0 %)
sin ninguna dimensión: 1722 (29.1 %)


## 8. Estructura del archivo de salida

In [13]:
print(f"{len(fragmentos)} filas × {len(fragmentos.columns)} columnas")
fragmentos[["fragmento_id", "establecimiento", "zona", "rating",
            "texto", "texto_limpio", "n_dim"]].head(4)

5913 filas × 20 columnas


,fragmento_id,establecimiento,zona,rating,texto,texto_limpio,n_dim
0,ChZDSUhNMG9nS0VJQ0FnSUM5aHFMOElnEAE_01,...Quieta Margarita!,fuera,5,Excelente sabor y excelente atención,excelente sabor excelente atención,2
1,ChZDSUhNMG9nS0VJQ0FnSUM5aHFMOElnEAE_02,...Quieta Margarita!,fuera,5,Felicitaciones,felicitaciones,0
2,ChZDSUhNMG9nS0VJQ0FnSURqcXJXS1lnEAE_01,...Quieta Margarita!,fuera,5,"Un lugar muy agradable, atención maravillosa , platos variados y deliciosos",lugar agradable atención maravillosa platos variados deliciosos,3
3,ChZDSUhNMG9nS0VJQ0FnSURqcXJXS1lnEAE_02,...Quieta Margarita!,fuera,5,Lo recomiendo mucho,recomiendo,0


## Hallazgos

1. **El embudo cierra sin pérdidas inesperadas.** De 8.451 reseñas quedan 2.432 en
   español con al menos 50 caracteres, exactamente la cifra esperada, y tras eliminar 2
   duplicados reales se obtienen **5.913 fragmentos de 2.430 reseñas**. Ninguna reseña
   pierde todos sus fragmentos.

2. **Recalcular las dimensiones por fragmento no pierde señal.** Al reagregarlas a nivel
   de reseña coinciden con la medición validada del proyecto original con una diferencia
   máxima del 1,0 %. Esto autoriza la decisión de no heredarlas.

3. **Segmentar por cláusula mejora la pureza temática.** El 42,0 % de los fragmentos
   tiene exactamente una dimensión activa, contra el 20,9 % de las reseñas completas.
   El AMI del objetivo 3 dispone entonces del doble de corpus limpio.

4. **El 29,1 % de los fragmentos no activa ninguna dimensión.** Casi un tercio del corpus
   queda fuera del esquema a priori. Son los fragmentos donde el modelado de tópicos
   puede aportar algo que el diccionario no ve, y son exactamente los que
   `sentimiento_fragmentos.csv` habría descartado.

5. **Dos correcciones fueron necesarias y quedan documentadas:** la regla de enmascarado
   del proyecto original, aplicada globalmente, producía un 14,0 % de falsos positivos;
   y quitar las negaciones del texto de c-TF-IDF invertía la etiqueta de los tópicos
   negativos.

---

*Este notebook lee de `data/processed/` los resultados ya calculados. Los scripts que
los producen están en `scripts/` y las decisiones que los sustentan, con sus cifras, en
`docs/bitacora.md` y `docs/decisiones_metodologicas.md`.*